In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import pickle
import joblib

# Load player data
df = pd.read_csv('/content/drive/My Drive/Football/df_clean_archetype.csv')

# Load archetype names
with open('/content/drive/My Drive/Football/archetype_names.pkl', 'rb') as f:
    archetype_names = pickle.load(f)

# Load models (for similarity calculations)
kmeans = joblib.load('/content/drive/My Drive/Football/kmeans_model.pkl')
pca = joblib.load('/content/drive/My Drive/Football/pca_model.pkl')
scaler = joblib.load('/content/drive/My Drive/Football/scaler.pkl')

print("Data loaded successfully")
print(f"Players: {len(df)}")
print(f"Columns: {df.columns.tolist()[:15]}")
print(f"Archetype names: {archetype_names}")

Data loaded successfully
Players: 14288
Columns: ['league', 'season', 'team', 'player', 'pos_', 'age_', 'Playing Time_MP', 'Playing Time_Starts', 'Playing Time_Min', 'Playing Time_90s', 'Performance_Gls', 'Performance_Ast', 'Performance_G+A', 'Performance_G-PK', 'Performance_PK']
Archetype names: {0: 'Central Midfielder', 1: 'Center Back', 2: 'Complete Forward', 3: 'Ball-Playing Defender', 4: 'Poacher', 5: 'Wide Midfielder', 6: 'Attacking Midfielder'}


In [ ]:
from difflib import get_close_matches

# Create unique players dataframe (one row per player)
unique_players = df.drop_duplicates(subset=['player']).copy()

# Add nickname mapping for common short names
nickname_map = {
    'KDB': 'Kevin De Bruyne',
    'VVD': 'Virgil van Dijk',
    'Messi': 'Lionel Messi',
    'Ronaldo': 'Cristiano Ronaldo',
    'Salah': 'Mohamed Salah',
    'Mane': 'Sadio Mane',
    'Firmino': 'Roberto Firmino',
    'Auba': 'Pierre-Emerick Aubameyang',
    'Laca': 'Alexandre Lacazette',
    'Kante': 'N Golo Kante',
    'Hazard': 'Eden Hazard',
    'Sane': 'Leroy Sane',
    'Ozil': 'Mesut Ozil',
    'Kane': 'Harry Kane',
    'Son': 'Son Heung-Min',
    'Sterling': 'Raheem Sterling',
    'Bernardo': 'Bernardo Silva',
    'Silva': 'David Silva',
    'Walker': 'Kyle Walker',
    'Stones': 'John Stones',
    'Foden': 'Phil Foden',
    'Grealish': 'Jack Grealish',
    'Haaland': 'Erling Haaland',
    'Alvarez': 'Julian Alvarez',
    'Rodri': 'Rodri',
    'Casemiro': 'Casemiro',
    'Fernandes': 'Bruno Fernandes',
    'Rashford': 'Marcus Rashford'
}

def get_player_archetype(player_name, fuzzy_match=True):
    """
    Returns the tactical archetype of a given player.

    Args:
        player_name (str): Name of the player
        fuzzy_match (bool): If True, tries substring matching

    Returns:
        str: Archetype name or suggestions
    """
    original_input = player_name

    # Check nickname map first
    if player_name in nickname_map:
        player_name = nickname_map[player_name]

    # Try exact match first
    player_row = unique_players[unique_players['player'] == player_name]

    # If exact match fails, try substring match
    if player_row.empty and fuzzy_match:
        # Find players whose name contains the search term (case insensitive)
        matches = unique_players[unique_players['player'].str.contains(player_name, case=False, na=False)]

        if len(matches) == 1:
            player_row = matches
        elif len(matches) > 1:
            match_names = matches['player'].tolist()[:5]
            return f"Multiple players found for '{original_input}'. Did you mean: {', '.join(match_names)}?"
        else:
            # Try fuzzy matching as last resort
            all_players = unique_players['player'].tolist()
            fuzzy_matches = get_close_matches(player_name, all_players, n=3, cutoff=0.5)
            fuzzy_matches = list(dict.fromkeys(fuzzy_matches))  # Remove duplicates

            if fuzzy_matches:
                return f"Player '{original_input}' not found. Did you mean: {', '.join(fuzzy_matches)}?"
            else:
                return f"Player '{original_input}' not found in database."

    if player_row.empty:
        return f"Player '{original_input}' not found in database."

    archetype_id = int(player_row['archetype'].iloc[0])
    archetype = archetype_names[archetype_id]

    # Add additional context if nickname was used
    if original_input in nickname_map:
        return f"{archetype} (matched '{original_input}' to {player_name})"

    return archetype


# Test all scenarios

print("\n1. Exact match:")
print("   'Mohamed Salah' ->", get_player_archetype('Mohamed Salah'))

print("\n2. Nickname match:")
print("   'Salah' ->", get_player_archetype('Salah'))
print("   'KDB' ->", get_player_archetype('KDB'))
print("   'VVD' ->", get_player_archetype('VVD'))

print("\n3. Substring match (single result):")
print("   'Xhaka' ->", get_player_archetype('Xhaka'))

print("\n4. Substring match (multiple results):")
print("   'Alex' ->", get_player_archetype('Alex'))

print("\n5. Player not found:")
print("   'FakePlayer' ->", get_player_archetype('FakePlayer'))


1. Exact match:
   'Mohamed Salah' -> Complete Forward

2. Nickname match:
   'Salah' -> Complete Forward (matched 'Salah' to Mohamed Salah)
   'KDB' -> Complete Forward (matched 'KDB' to Kevin De Bruyne)
   'VVD' -> Center Back (matched 'VVD' to Virgil van Dijk)

3. Substring match (single result):
   'Xhaka' -> Central Midfielder

4. Substring match (multiple results):
   'Alex' -> Multiple players found for 'Alex'. Did you mean: Alex Iwobi, Alexandre Lacazette, Alexis Sánchez, Alex Pritchard, Alex Oxlade-Chamberlain?

5. Player not found:
   'FakePlayer' -> Player 'FakePlayer' not found. Did you mean: Gelabert, Falaye Sacko, Daniel Baier?


In [ ]:
def find_similar_scouting(player_name, top_n=5, weights=None):
    """
    Finds similar players using weighted combination of PCA and key stats.
    """
    # Find the player
    player_row = unique_players[unique_players['player'] == player_name]

    if player_row.empty:
        return f"Player '{player_name}' not found."

    # Default weights (balanced for scouting)
    if weights is None:
        weights = {
            'pca': 0.4,
            'goals': 0.15,
            'tackles': 0.15,
            'passing': 0.15,
            'creativity': 0.15
        }

    # Get target player's values
    pc1_target = player_row['PC1'].iloc[0]
    pc2_target = player_row['PC2'].iloc[0]

    # Get stats (handle missing columns)
    stats_map = {}
    for stat in ['Performance_Gls_norm', 'Performance_TklW_norm', 'Total_Cmp%_norm', 'Creativity_Score_norm']:
        stats_map[stat] = player_row[stat].iloc[0] if stat in player_row.columns else 0

    # Calculate similarity for all players
    temp_df = unique_players.copy()

    similarities = []
    for idx, row in temp_df.iterrows():
        if row['player'] == player_name:
            similarities.append(0)
            continue

        # PCA similarity
        pca_dist = np.sqrt((row['PC1'] - pc1_target)**2 + (row['PC2'] - pc2_target)**2)
        pca_sim = max(0, 100 - (pca_dist * 15))  # Max PCA similarity 100, min 0

        # Stats similarities
        goals_sim = 100 - min(100, abs(row.get('Performance_Gls_norm', 0) - stats_map['Performance_Gls_norm']) * 30)
        tackles_sim = 100 - min(100, abs(row.get('Performance_TklW_norm', 0) - stats_map['Performance_TklW_norm']) * 30)
        passing_sim = 100 - min(100, abs(row.get('Total_Cmp%_norm', 0) - stats_map['Total_Cmp%_norm']) * 30)
        creativity_sim = 100 - min(100, abs(row.get('Creativity_Score_norm', 0) - stats_map['Creativity_Score_norm']) * 30)

        # Weighted average
        total_sim = (
            weights['pca'] * pca_sim +
            weights['goals'] * goals_sim +
            weights['tackles'] * tackles_sim +
            weights['passing'] * passing_sim +
            weights['creativity'] * creativity_sim
        )

        similarities.append(total_sim)

    temp_df['similarity'] = similarities

    # Get top N
    similar = temp_df[temp_df['player'] != player_name]
    similar = similar.nlargest(top_n, 'similarity')

    # Prepare results
    results = []
    for _, row in similar.iterrows():
        archetype = archetype_names[int(row['archetype'])]
        results.append({
            'name': row['player'],
            'archetype': archetype,
            'similarity': round(row['similarity'], 1),
            'team': row.get('team', 'Unknown')
        })

    return results

# Test the corrected function

print("\nFinding players similar to Rodri:")
scouting_results = find_similar_scouting('Rodri', top_n=5)
if isinstance(scouting_results, list):
    for i, player in enumerate(scouting_results, 1):
        print(f"{i}. {player['name']} - {player['archetype']}")
        print(f"   Similarity: {player['similarity']}%")
        print(f"   Team: {player['team']}")
        print()
else:
    print(scouting_results)


Finding players similar to Rodri:
1. Nicolás Otamendi - Central Midfielder
   Similarity: 89.5%
   Team: Manchester City

2. Adrien Rabiot - Central Midfielder
   Similarity: 88.9%
   Team: Paris S-G

3. Geoffrey Kondogbia - Central Midfielder
   Similarity: 87.6%
   Team: Valencia

4. Milan Badelj - Central Midfielder
   Similarity: 86.5%
   Team: Fiorentina

5. Ivan Rakitić - Central Midfielder
   Similarity: 85.8%
   Team: Barcelona



In [ ]:
def get_players_by_archetype(archetype_name, limit=10):
    """
    Returns a list of players belonging to a specific archetype.

    Args:
        archetype_name (str): Name of the archetype (e.g., 'Complete Forward')
        limit (int): Maximum number of players to return (default 10)

    Returns:
        list: List of player names and their details
    """
    # Find archetype ID from name
    archetype_id = None
    for aid, aname in archetype_names.items():
        if aname.lower() == archetype_name.lower():
            archetype_id = aid
            break

    if archetype_id is None:
        # Show available archetypes
        available = list(archetype_names.values())
        return f"Archetype '{archetype_name}' not found. Available archetypes: {', '.join(available)}"

    # Get players in this archetype
    players_in_archetype = unique_players[unique_players['archetype'] == archetype_id]

    if len(players_in_archetype) == 0:
        return f"No players found in archetype '{archetype_name}'."

    # Get top players (you can add sorting logic later)
    result = []
    for _, row in players_in_archetype.head(limit).iterrows():
        result.append({
            'name': row['player'],
            'team': row.get('team', 'Unknown'),
            'nationality': row.get('nation_', 'Unknown'),
            'primary_position': row.get('pos_', 'Unknown')
        })

    return result

# Test Function 4


print("\n1. Complete Forwards:")
forwards = get_players_by_archetype('Complete Forward', limit=10)
if isinstance(forwards, list):
    for i, player in enumerate(forwards, 1):
        print(f"{i}. {player['name']} - {player['team']} ({player['nationality']})")
else:
    print(forwards)

print("\n2. Center Backs:")
defenders = get_players_by_archetype('Center Back', limit=10)
if isinstance(defenders, list):
    for i, player in enumerate(defenders, 1):
        print(f"{i}. {player['name']} - {player['team']} ({player['nationality']})")
else:
    print(defenders)

print("\n3. Invalid archetype:")
invalid = get_players_by_archetype('Striker')
print(invalid)


1. Complete Forwards:
1. Pascal Groß - Brighton (GER)
2. Eden Hazard - Chelsea (BEL)
3. Wilfried Zaha - Crystal Palace (CIV)
4. Riyad Mahrez - Leicester City (ALG)
5. Mohamed Salah - Liverpool (EGY)
6. Roberto Firmino - Liverpool (BRA)
7. Sadio Mané - Liverpool (SEN)
8. David Silva - Manchester City (ESP)
9. Kevin De Bruyne - Manchester City (BEL)
10. Leroy Sané - Manchester City (GER)

2. Center Backs:
1. Ainsley Maitland-Niles - Arsenal (ENG)
2. Calum Chambers - Arsenal (ENG)
3. Mohamed Elneny - Arsenal (EGY)
4. Rob Holding - Arsenal (ENG)
5. Harry Arter - Bournemouth (IRL)
6. Marc Pugh - Bournemouth (ENG)
7. Steve Cook - Bournemouth (ENG)
8. Beram Kayal - Brighton (ISR)
9. Bruno - Brighton (ESP)
10. Ezequiel Schelotto - Brighton (ITA)

3. Invalid archetype:
Archetype 'Striker' not found. Available archetypes: Central Midfielder, Center Back, Complete Forward, Ball-Playing Defender, Poacher, Wide Midfielder, Attacking Midfielder


In [ ]:
# Country code mapping
country_names = {
    'ENG': 'England',
    'EGY': 'Egypt',
    'FRA': 'France',
    'ESP': 'Spain',
    'GER': 'Germany',
    'ITA': 'Italy',
    'BEL': 'Belgium',
    'NED': 'Netherlands',
    'POR': 'Portugal',
    'BRA': 'Brazil',
    'ARG': 'Argentina',
    'URU': 'Uruguay',
    'COL': 'Colombia',
    'CHI': 'Chile',
    'SEN': 'Senegal',
    'NGA': 'Nigeria',
    'CMR': 'Cameroon',
    'GHA': 'Ghana',
    'CIV': 'Ivory Coast',
    'MAR': 'Morocco',
    'ALG': 'Algeria',
    'TUN': 'Tunisia',
    'KOR': 'South Korea',
    'JPN': 'Japan',
    'AUS': 'Australia',
    'USA': 'United States',
    'MEX': 'Mexico',
    'CAN': 'Canada',
    'POL': 'Poland',
    'CRO': 'Croatia',
    'SRB': 'Serbia',
    'SUI': 'Switzerland',
    'SWE': 'Sweden',
    'DEN': 'Denmark',
    'NOR': 'Norway',
    'FIN': 'Finland',
    'AUT': 'Austria',
    'CZE': 'Czech Republic',
    'GRE': 'Greece',
    'TUR': 'Turkey',
    'RUS': 'Russia',
    'UKR': 'Ukraine',
    'ROU': 'Romania',
    'BUL': 'Bulgaria',
    'HUN': 'Hungary',
    'IRL': 'Ireland',
    'SCO': 'Scotland',
    'WAL': 'Wales',
    'NIR': 'Northern Ireland',
}

def get_player_nationality(player_name):
    """
    Returns the full country name for a player.
    """
    player_row = unique_players[unique_players['player'] == player_name]

    if player_row.empty:
        return "Unknown"

    code = player_row['nation_'].iloc[0]
    full_name = country_names.get(code, code)

    return full_name

# Test
print(get_player_nationality('Mohamed Salah'))  # Should print Egypt
print(get_player_nationality('Harry Kane'))    # Should print England
print(get_player_nationality('Kevin De Bruyne')) # Should print Belgium

Egypt
England
Belgium


In [ ]:
def compare_players(player1_name, player2_name):
    """
    Compares two players side by side.
    """
    # Get both players
    player1 = unique_players[unique_players['player'] == player1_name]
    player2 = unique_players[unique_players['player'] == player2_name]

    if player1.empty:
        return f"Player '{player1_name}' not found."
    if player2.empty:
        return f"Player '{player2_name}' not found."

    # Get archetypes
    arch1_id = int(player1['archetype'].iloc[0])
    arch2_id = int(player2['archetype'].iloc[0])
    arch1 = archetype_names[arch1_id]
    arch2 = archetype_names[arch2_id]

    # Get nationalities using full names
    nat1 = get_player_nationality(player1_name)
    nat2 = get_player_nationality(player2_name)

    # Get key stats
    stats = ['Performance_Gls_norm', 'Performance_TklW_norm', 'Total_Cmp%_norm',
             'Progression_PrgC_norm', 'Creativity_Score_norm']
    stats_labels = ['Goals', 'Tackles', 'Passing', 'Carrying', 'Creativity']

    result = {
        'player1': {
            'name': player1_name,
            'archetype': arch1,
            'nationality': nat1,
            'team': player1['team'].iloc[0],
            'stats': {}
        },
        'player2': {
            'name': player2_name,
            'archetype': arch2,
            'nationality': nat2,
            'team': player2['team'].iloc[0],
            'stats': {}
        }
    }

    # Add stats
    for i, stat in enumerate(stats):
        if stat in player1.columns:
            result['player1']['stats'][stats_labels[i]] = round(player1[stat].iloc[0], 2)
            result['player2']['stats'][stats_labels[i]] = round(player2[stat].iloc[0], 2)

    return result

# Test
print("COMPARING MOHAMED SALAH vs KEVIN DE BRUYNE")
print("="*50)

comparison = compare_players('Mohamed Salah', 'Kevin De Bruyne')
if isinstance(comparison, dict):
    p1 = comparison['player1']
    p2 = comparison['player2']

    print(f"\n{p1['name']} ({p1['archetype']} - {p1['nationality']} - {p1['team']})")
    print("vs")
    print(f"{p2['name']} ({p2['archetype']} - {p2['nationality']} - {p2['team']})")

    print("\nStats Comparison (Normalized Scores):")
    for stat in p1['stats'].keys():
        print(f"  {stat}: {p1['stats'][stat]} vs {p2['stats'][stat]}")
else:
    print(comparison)

COMPARING MOHAMED SALAH vs KEVIN DE BRUYNE

Mohamed Salah (Complete Forward - Egypt - Liverpool)
vs
Kevin De Bruyne (Complete Forward - Belgium - Manchester City)

Stats Comparison (Normalized Scores):
  Goals: 7.41 vs 1.37
  Tackles: -1.11 vs 1.25
  Passing: -0.45 vs 0.19
  Carrying: 1.82 vs 4.54
  Creativity: 7.72 vs 24.93


In [ ]:
def get_player_stats_summary(player_name):
    """
    Returns a readable summary of a player's key stats.
    """
    player_row = unique_players[unique_players['player'] == player_name]

    if player_row.empty:
        return f"Player '{player_name}' not found."

    archetype = get_player_archetype(player_name)
    nationality = get_player_nationality(player_name)

    # Get stats
    goals = player_row['Performance_Gls_norm'].iloc[0] if 'Performance_Gls_norm' in player_row else 0
    tackles = player_row['Performance_TklW_norm'].iloc[0] if 'Performance_TklW_norm' in player_row else 0
    passing = player_row['Total_Cmp%_norm'].iloc[0] if 'Total_Cmp%_norm' in player_row else 0
    creativity = player_row['Creativity_Score_norm'].iloc[0] if 'Creativity_Score_norm' in player_row else 0

    # Convert to readable interpretation
    def interpret(value, high_is_good=True):
        if high_is_good:
            if value > 2: return "Elite"
            elif value > 0.5: return "Above Average"
            elif value > -0.5: return "Average"
            else: return "Below Average"
        else:
            if value < -1: return "Low"
            elif value < 0: return "Below Average"
            else: return "Average"

    summary = f"""
Player: {player_name}
Nationality: {nationality}
Archetype: {archetype}

Key Stats:
- Goals: {interpret(goals)} ({round(goals, 2)})
- Tackles: {interpret(tackles)} ({round(tackles, 2)})
- Passing: {interpret(passing)} ({round(passing, 2)})
- Creativity: {interpret(creativity)} ({round(creativity, 2)})
"""
    return summary

# Test
print(get_player_stats_summary('Mohamed Salah'))


Player: Mohamed Salah
Nationality: Egypt
Archetype: Complete Forward

Key Stats:
- Goals: Elite (7.41)
- Tackles: Below Average (-1.11)
- Passing: Average (-0.45)
- Creativity: Elite (7.72)



In [ ]:
!pip install langchain langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.1 MB/s eta 0:00:00


In [ ]:


import os
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"] = "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"

# Option 1: High quality (recommended for football analysis)
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.3)

# Option 2: Fast and efficient
# llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.3)

# Option 3: Newest model
# llm = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct", temperature=0.3)

In [ ]:
from langchain.tools import tool

@tool
def get_player_archetype_tool(player_name: str) -> str:
    """Get the tactical archetype of a player."""
    return get_player_archetype(player_name)

@tool
def get_player_stats_summary_tool(player_name: str) -> str:
    """Get a summary of a player's key stats."""
    return get_player_stats_summary(player_name)

@tool
def compare_players_tool(player1: str, player2: str) -> str:
    """Compare two players side by side."""
    result = compare_players(player1, player2)
    if isinstance(result, dict):
        p1 = result['player1']
        p2 = result['player2']
        return f"{p1['name']} ({p1['archetype']}) vs {p2['name']} ({p2['archetype']})\nStats: {p1['stats']} vs {p2['stats']}"
    return str(result)

@tool
def find_similar_players_tool(player_name: str, top_n: int = 5) -> str:
    """Find players with similar playing style."""
    results = find_similar_scouting(player_name, top_n)
    if isinstance(results, list):
        output = f"Players similar to {player_name}:\n"
        for i, p in enumerate(results, 1):
            output += f"{i}. {p['name']} ({p['archetype']}) - {p['similarity']}% similar\n"
        return output
    return str(results)

@tool
def get_players_by_archetype_tool(archetype_name: str, limit: int = 10) -> str:
    """List players in a specific archetype."""
    results = get_players_by_archetype(archetype_name, limit)
    if isinstance(results, list):
        output = f"Players in {archetype_name}:\n"
        for i, p in enumerate(results, 1):
            output += f"{i}. {p['name']} - {p['team']}\n"
        return output
    return str(results)

tools = [
    get_player_archetype_tool,
    get_player_stats_summary_tool,
    compare_players_tool,
    find_similar_players_tool,
    get_players_by_archetype_tool,
]

In [ ]:
response = llm.invoke("What is the capital of France?")
print(response.content)

The capital of France is Paris.


In [ ]:
!pip install langgraph langchain-groq

import os
from langchain_groq import ChatGroq
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool

os.environ["GROQ_API_KEY"] = "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"

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.3)

# Define your tools (assuming functions exist)
@tool
def get_player_archetype_tool(player_name: str) -> str:
    """Get the tactical archetype of a player (e.g., Complete Forward, Central Midfielder)."""
    return get_player_archetype(player_name)

@tool
def get_player_stats_summary_tool(player_name: str) -> str:
    """Get a summary of a player's key stats including goals, tackles, passing, and creativity."""
    return get_player_stats_summary(player_name)

tools = [get_player_archetype_tool, get_player_stats_summary_tool]

# Create agent
agent = create_react_agent(model=llm, tools=tools)

# Test
response = agent.invoke({"messages": [("user", "What archetype is Mohamed Salah?")]})
print(response["messages"][-1].content)

/tmp/ipykernel_2067/4262242657.py:26: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(model=llm, tools=tools)


Mohamed Salah's tactical archetype is a Complete Forward.


In [ ]:
import os
folder_path = '/content/drive/My Drive/Football/'
print(os.listdir(folder_path))

['Top5_League_Players_2017to2024_dataset.csv', 'cleaned_football_data.csv', 'football_features_normalized.csv', 'pca_results.csv', 'kmeans_model.pkl', 'pca_model.pkl', 'scaler.pkl', 'features_list.pkl', 'archetype_names.pkl', 'player_data_with_nation.csv', 'df_clean_archetype.csv', 'Football_Tactical_Fingerprint.ipynb', 'football_agent_tools.ipynb']
